# Module 6: Probability Distributions & Statistics

Every AI model works with probability distributions. This module covers the distributions you'll encounter daily.

### 🎯 What you'll learn:
- Key discrete distributions (Bernoulli, Binomial, Poisson)
- Key continuous distributions (Gaussian, Exponential, Beta, Gamma)
- Multivariate Gaussian distribution
- Maximum Likelihood Estimation (MLE)
- Maximum A Posteriori (MAP) estimation
- Hypothesis testing and confidence intervals

### 🤖 Why it matters for AI:
- **Gaussian** is everywhere: noise, priors, latent spaces
- **MLE** is how we train most ML models (minimizing cross-entropy = MLE!)
- **Beta/Dirichlet** are priors for probabilities
- **Multivariate Gaussian** used in Gaussian processes, VAEs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.style.use('seaborn-v0_8-darkgrid')

---
## 1. Discrete Distributions

### Bernoulli($p$)
Single binary trial: $P(X=1) = p$, $P(X=0) = 1-p$
- $E[X] = p$, $\text{Var}(X) = p(1-p)$

### Binomial($n, p$)
Number of successes in $n$ independent Bernoulli trials:
$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$
- $E[X] = np$, $\text{Var}(X) = np(1-p)$

### Poisson($\lambda$)
Count of events in a fixed interval:
$$P(X=k) = \frac{\lambda^k e^{-\lambda}}{k!}$$
- $E[X] = \lambda$, $\text{Var}(X) = \lambda$

### Categorical / Multinomial
Generalization of Bernoulli/Binomial to $K$ categories:
$$P(X=k) = p_k, \quad \sum_k p_k = 1$$

🤖 **Softmax output** of a classifier is a categorical distribution!

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Binomial with different parameters
for n, p, color in [(10, 0.3, '#E74C3C'), (20, 0.5, '#3498DB'), (50, 0.7, '#2ECC71')]:
    k = np.arange(0, n+1)
    axes[0].bar(k, stats.binom.pmf(k, n, p), alpha=0.5, label=f'n={n}, p={p}', color=color)
axes[0].set_title('Binomial Distribution', fontsize=14)
axes[0].legend(); axes[0].set_xlabel('k'); axes[0].set_ylabel('P(X=k)')

# Poisson with different λ
k = np.arange(0, 20)
for lam, color in [(1, '#E74C3C'), (4, '#3498DB'), (10, '#2ECC71')]:
    axes[1].bar(k, stats.poisson.pmf(k, lam), alpha=0.5, label=f'λ={lam}', color=color, width=0.8)
axes[1].set_title('Poisson Distribution', fontsize=14)
axes[1].legend(); axes[1].set_xlabel('k')

# Categorical (softmax output example)
classes = ['cat', 'dog', 'bird', 'fish']
probs = [0.6, 0.25, 0.1, 0.05]  # Softmax output
axes[2].bar(classes, probs, color=['#E74C3C', '#3498DB', '#2ECC71', '#F39C12'])
axes[2].set_title('Categorical (Softmax Output)', fontsize=14)
axes[2].set_ylabel('Probability')

plt.tight_layout()
plt.show()

---
## 2. The Gaussian (Normal) Distribution

The **most important distribution in all of ML**:

$$f(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

- $\mu$ = mean (center), $\sigma^2$ = variance (spread)
- $E[X] = \mu$, $\text{Var}(X) = \sigma^2$
- 68-95-99.7 rule: 68% within 1σ, 95% within 2σ, 99.7% within 3σ

### Why is it everywhere?
1. **CLT**: Sum of many random variables → Gaussian
2. **Maximum entropy**: Among distributions with known mean/variance, Gaussian has max entropy
3. **Conjugate prior**: Gaussian prior + Gaussian likelihood = Gaussian posterior

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.linspace(-6, 6, 300)
# Different means and variances
params = [(0, 1, '#E74C3C'), (0, 2, '#3498DB'), (2, 0.5, '#2ECC71'), (-1, 1.5, '#9B59B6')]
for mu, sigma, color in params:
    axes[0].plot(x, stats.norm.pdf(x, mu, sigma), color=color, lw=2, label=f'μ={mu}, σ={sigma}')
axes[0].set_title('Gaussian PDF', fontsize=14)
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 68-95-99.7 rule
x_fill = np.linspace(-4, 4, 300)
pdf = stats.norm.pdf(x_fill)
axes[1].plot(x_fill, pdf, 'k-', lw=2)
axes[1].fill_between(x_fill, pdf, where=(np.abs(x_fill) <= 1), alpha=0.4, color='#E74C3C', label='68% (±1σ)')
axes[1].fill_between(x_fill, pdf, where=(np.abs(x_fill) <= 2) & (np.abs(x_fill) > 1), alpha=0.3, color='#3498DB', label='95% (±2σ)')
axes[1].fill_between(x_fill, pdf, where=(np.abs(x_fill) <= 3) & (np.abs(x_fill) > 2), alpha=0.2, color='#2ECC71', label='99.7% (±3σ)')
axes[1].set_title('68-95-99.7 Rule', fontsize=14)
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Other Continuous Distributions

### Exponential($\lambda$)
Time between events: $f(x) = \lambda e^{-\lambda x}$ for $x \geq 0$
- **Memoryless**: $P(X > s+t | X > s) = P(X > t)$

### Beta($\alpha, \beta$)
Distribution on $[0, 1]$ — perfect for modeling probabilities!
$$f(x) = \frac{x^{\alpha-1}(1-x)^{\beta-1}}{B(\alpha, \beta)}$$
- **Conjugate prior** for Bernoulli/Binomial

### Gamma($\alpha, \beta$)
Generalization of Exponential. Models positive continuous values.

### Dirichlet($\boldsymbol{\alpha}$)
Multivariate Beta — distribution over probability vectors.
- **Conjugate prior** for Categorical/Multinomial
- Used in **LDA** topic modeling

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Exponential
x = np.linspace(0, 5, 200)
for lam, color in [(0.5, '#E74C3C'), (1, '#3498DB'), (2, '#2ECC71')]:
    axes[0].plot(x, stats.expon.pdf(x, scale=1/lam), color=color, lw=2, label=f'λ={lam}')
axes[0].set_title('Exponential Distribution', fontsize=14); axes[0].legend()

# Beta
x = np.linspace(0.001, 0.999, 200)
for a, b, color in [(0.5, 0.5, '#E74C3C'), (2, 5, '#3498DB'), (5, 2, '#2ECC71'), (2, 2, '#9B59B6')]:
    axes[1].plot(x, stats.beta.pdf(x, a, b), color=color, lw=2, label=f'α={a}, β={b}')
axes[1].set_title('Beta Distribution (prior for probabilities)', fontsize=14); axes[1].legend()

# Gamma
x = np.linspace(0.01, 15, 200)
for a, b, color in [(1, 2, '#E74C3C'), (2, 2, '#3498DB'), (5, 1, '#2ECC71'), (9, 0.5, '#9B59B6')]:
    axes[2].plot(x, stats.gamma.pdf(x, a, scale=b), color=color, lw=2, label=f'α={a}, β={b}')
axes[2].set_title('Gamma Distribution', fontsize=14); axes[2].legend()

for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Multivariate Gaussian

$$f(\mathbf{x}) = \frac{1}{(2\pi)^{d/2}|\boldsymbol{\Sigma}|^{1/2}} \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T \boldsymbol{\Sigma}^{-1} (\mathbf{x}-\boldsymbol{\mu})\right)$$

- $\boldsymbol{\mu} \in \mathbb{R}^d$ = mean vector
- $\boldsymbol{\Sigma} \in \mathbb{R}^{d \times d}$ = covariance matrix (symmetric, positive definite)

### 🤖 AI Connection:
- **VAE latent space** is typically $N(\mathbf{0}, \mathbf{I})$
- **Gaussian Mixture Models** for clustering
- **Gaussian Processes** for uncertainty-aware regression

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

covs = [
    ([[1, 0], [0, 1]], 'Independent (Σ=I)'),
    ([[1, 0.8], [0.8, 1]], 'Positive correlation'),
    ([[2, -0.5], [-0.5, 0.5]], 'Different variances')
]

for ax, (cov, title) in zip(axes, covs):
    samples = np.random.multivariate_normal([0, 0], cov, 1000)
    ax.scatter(samples[:, 0], samples[:, 1], alpha=0.3, s=10, c='#3498DB')
    
    # Draw contours
    X, Y = np.meshgrid(np.linspace(-4, 4, 100), np.linspace(-4, 4, 100))
    pos = np.dstack((X, Y))
    rv = stats.multivariate_normal([0, 0], cov)
    ax.contour(X, Y, rv.pdf(pos), levels=5, colors='red', alpha=0.7)
    ax.set_title(title, fontsize=13); ax.set_aspect('equal')
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.grid(True, alpha=0.3)

plt.suptitle('Multivariate Gaussian with Different Covariance Matrices', fontsize=16, y=1.02)
plt.tight_layout(); plt.show()

---
## 5. Maximum Likelihood Estimation (MLE)

Given data $\{x_1, \ldots, x_n\}$, find parameters $\theta$ that maximize:

$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta \prod_{i=1}^{n} P(x_i | \theta) = \arg\max_\theta \sum_{i=1}^{n} \log P(x_i | \theta)$$

### 🤖 Why MLE = Training Neural Networks:
- **Cross-entropy loss** = negative log-likelihood
- **MSE loss** = MLE assuming Gaussian noise
- Minimizing loss = Maximizing likelihood!

In [ ]:
# MLE for Gaussian: estimate μ and σ from data
np.random.seed(42)
true_mu, true_sigma = 5.0, 2.0
data = np.random.normal(true_mu, true_sigma, 100)

# MLE estimates (closed-form for Gaussian)
mu_mle = np.mean(data)
sigma_mle = np.std(data)  # MLE uses 1/n, not 1/(n-1)

print(f"True: μ={true_mu}, σ={true_sigma}")
print(f"MLE:  μ={mu_mle:.4f}, σ={sigma_mle:.4f}")

# Visualize the likelihood surface
mus = np.linspace(3, 7, 100)
sigmas = np.linspace(1, 3.5, 100)
MU, SIGMA = np.meshgrid(mus, sigmas)

# Log-likelihood
log_lik = np.zeros_like(MU)
for i in range(len(sigmas)):
    for j in range(len(mus)):
        log_lik[i, j] = np.sum(stats.norm.logpdf(data, MU[i, j], SIGMA[i, j]))

fig, ax = plt.subplots(figsize=(10, 7))
cs = ax.contourf(MU, SIGMA, log_lik, levels=30, cmap='viridis')
ax.plot(mu_mle, sigma_mle, 'r*', markersize=20, label=f'MLE ({mu_mle:.2f}, {sigma_mle:.2f})')
ax.plot(true_mu, true_sigma, 'w*', markersize=15, label=f'True ({true_mu}, {true_sigma})')
ax.set_xlabel('μ', fontsize=14); ax.set_ylabel('σ', fontsize=14)
ax.set_title('Log-Likelihood Surface', fontsize=16)
ax.legend(fontsize=12); plt.colorbar(cs); plt.show()

---
## 6. Maximum A Posteriori (MAP) Estimation

MAP adds a **prior** to MLE:
$$\hat{\theta}_{\text{MAP}} = \arg\max_\theta P(\theta | \text{data}) = \arg\max_\theta [\log P(\text{data}|\theta) + \log P(\theta)]$$

### MLE vs MAP:
| | MLE | MAP |
|---|-----|-----|
| Prior | None | Yes |
| Overfitting | More prone | Regularized |
| Equivalent to | Cross-entropy loss | Loss + regularization |

### 🤖 AI Connection:
- **L2 regularization** (weight decay) = MAP with Gaussian prior on weights
- **L1 regularization** = MAP with Laplace prior on weights

In [ ]:
# MAP estimation: Coin flip with prior
# Observed: 3 heads in 3 flips (MLE would say P(heads)=1!)
n_heads, n_total = 3, 3

theta = np.linspace(0.01, 0.99, 200)

# Likelihood: Binomial
likelihood = theta**n_heads * (1-theta)**(n_total-n_heads)

# Prior: Beta(2, 2) — slight preference for fair coin
prior = stats.beta.pdf(theta, 2, 2)

# Posterior ∝ likelihood × prior
posterior = likelihood * prior
posterior /= np.trapz(posterior, theta)  # Normalize

mle = theta[np.argmax(likelihood)]
map_est = theta[np.argmax(posterior)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(theta, likelihood/np.max(likelihood), 'b--', lw=2, label='Likelihood (normalized)')
ax.plot(theta, prior/np.max(prior), 'g--', lw=2, label='Prior Beta(2,2)')
ax.plot(theta, posterior/np.max(posterior), 'r-', lw=3, label='Posterior')
ax.axvline(mle, color='blue', ls=':', label=f'MLE = {mle:.2f}')
ax.axvline(map_est, color='red', ls=':', label=f'MAP = {map_est:.2f}')
ax.set_xlabel('θ (P(heads))', fontsize=14); ax.set_ylabel('Density', fontsize=14)
ax.set_title('MLE vs MAP: Prior Prevents Overconfidence', fontsize=16)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.show()

print(f"MLE says P(heads) = {mle:.2f} (100%! Overfit!)")
print(f"MAP says P(heads) = {map_est:.2f} (more reasonable with prior)")

---
## 7. Hypothesis Testing

### Framework:
1. **Null hypothesis** $H_0$: No effect / default assumption
2. **Alternative** $H_1$: There IS an effect
3. Compute **test statistic** and **p-value**
4. Reject $H_0$ if p-value < significance level $\alpha$ (typically 0.05)

### p-value
Probability of seeing data this extreme (or more) IF $H_0$ is true.

### 🤖 AI Connection:
- **A/B testing** in production ML systems
- Determining if a model improvement is statistically significant

In [ ]:
# A/B test: Is model B better than model A?
np.random.seed(42)

# Model A accuracy on 200 test samples
model_a_scores = np.random.normal(0.85, 0.03, 200)
# Model B accuracy (slightly better)
model_b_scores = np.random.normal(0.87, 0.03, 200)

# Two-sample t-test
t_stat, p_value = stats.ttest_ind(model_a_scores, model_b_scores)

print("=== A/B Test: Is Model B better? ===")
print(f"Model A mean: {model_a_scores.mean():.4f}")
print(f"Model B mean: {model_b_scores.mean():.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Significant at α=0.05? {'YES ✅' if p_value < 0.05 else 'NO ❌'}")

---
## 8. Confidence Intervals

A **95% confidence interval** for the mean:
$$\bar{x} \pm z_{0.025} \cdot \frac{\sigma}{\sqrt{n}}$$

For unknown σ, use the t-distribution:
$$\bar{x} \pm t_{0.025, n-1} \cdot \frac{s}{\sqrt{n}}$$

### 🤖 AI Connection:
- Reporting model performance with confidence intervals
- Uncertainty quantification in predictions

In [ ]:
# Confidence interval for model accuracy
accuracies = np.random.normal(0.92, 0.02, 30)  # 30 evaluation runs

mean = np.mean(accuracies)
se = stats.sem(accuracies)  # Standard error
ci = stats.t.interval(0.95, df=len(accuracies)-1, loc=mean, scale=se)

print(f"Mean accuracy: {mean:.4f}")
print(f"95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"Report: {mean:.4f} ± {(ci[1]-ci[0])/2:.4f}")

---
## 9. Gaussian Mixture Models (Practical Example)

$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

A mixture of $K$ Gaussians. Trained using the **EM algorithm** (Expectation-Maximization).

In [ ]:
# Generate GMM data
np.random.seed(42)
n1 = np.random.multivariate_normal([2, 2], [[0.5, 0.2], [0.2, 0.5]], 200)
n2 = np.random.multivariate_normal([-1, -1], [[0.3, 0], [0, 0.8]], 200)
n3 = np.random.multivariate_normal([3, -2], [[0.4, -0.1], [-0.1, 0.3]], 150)
data = np.vstack([n1, n2, n3])

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(data[:, 0], data[:, 1], alpha=0.3, s=15, c='#3498DB')

# Show true cluster centers
for center, label in [([2,2], 'Cluster 1'), ([-1,-1], 'Cluster 2'), ([3,-2], 'Cluster 3')]:
    ax.plot(*center, 'r*', markersize=20)

ax.set_title('Gaussian Mixture Model Data (3 Clusters)', fontsize=16)
ax.grid(True, alpha=0.3)
plt.show()

---
## 10. Summary

| Distribution | Use in AI |
|-------------|----------|
| Gaussian | Noise, latent spaces, priors, everything |
| Bernoulli/Binomial | Binary classification, dropout |
| Categorical | Softmax output, language modeling |
| Beta | Prior for probabilities |
| Dirichlet | Prior for topic models |
| MLE | Training = maximizing likelihood |
| MAP | Training with regularization |

**Next: Optimization Theory — the engine that powers all ML training!** 🚀